In [1]:
import sys
print(sys.version)

3.12.5 | packaged by conda-forge | (main, Aug  8 2024, 18:24:51) [MSC v.1940 64 bit (AMD64)]


In [2]:
import sys

# Ensure pip is used from the same Python executable as the notebook
# The '-m' flag runs pip as a module, which is the best practice.
!{sys.executable} -m pip install selenium

  Using cached typing_extensions-4.15.0-py3-none-any.whl.metadata (3.3 kB)
Using cached typing_extensions-4.15.0-py3-none-any.whl (44 kB)
  Attempting uninstall: typing_extensions
    Found existing installation: typing-extensions None


error: uninstall-no-record-file

× Cannot uninstall typing-extensions None
╰─> The package's contents are unknown: no RECORD file was found for typing-extensions.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps typing-extensions==4.12.2


In [4]:
!pip show selenium

Name: selenium
Version: 4.37.0
Summary: Official Python bindings for Selenium WebDriver
Home-page: https://www.selenium.dev
Author: 
Author-email: 
License: Apache-2.0
Location: C:\Users\sezar\miniforge3\Lib\site-packages
Requires: certifi, trio, trio-websocket, typing_extensions, urllib3, websocket-client
Required-by: 


In [ ]:
import time
import csv
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.chrome.service import Service 
from selenium.common.exceptions import TimeoutException, NoSuchElementException, WebDriverException
import subprocess # NEW: For checking system commands

# --- Configuration ---
# Based on your example URL: P=7026c01&idColor=11#T=P&C=11
ITEM_CATEGORY = "P" # 'P' for Part, 'S' for Set, 'M' for Minifig - Defines the item type in the query string.
ITEM_ID = "7026c01" 
COLOR_ID = "11" 
ITEM_NAME = "Window 1 x 2 x 2 with Fixed Trans-Clear Glass" 

# --- URL CONSTRUCTION (Targets the Price Guide Tab) ---
ITEM_URL = f"https://www.bricklink.com/v2/catalog/catalogitem.page?P={ITEM_ID}&idColor={COLOR_ID}#T=P&C={COLOR_ID}"
OUTPUT_FILE = f"bricklink_sales_history_{ITEM_ID}_{COLOR_ID}.csv"


# --- CSS Selectors ---
# Selector for the link/button that reveals the price guide details.
PRICE_GUIDE_LINK_SELECTOR = "a.bl-item-detail-view-price-guide"
# Selector for the 'Past 6 Months Sales' table
SALES_TABLE_XPATH = "//div[@id='pastSixMonths']//table"
# Selector for the rows within the sales table
SALES_ROWS_XPATH = f"{SALES_TABLE_XPATH}/tbody/tr"

def get_chrome_version_from_registry():
    """Attempts to get Chrome version from Windows Registry (Windows-specific)."""
    try:
        # Command to query the DisplayVersion from the Chrome installation path
        # FIX: Changed single backslashes to double backslashes to fix SyntaxWarning
        command = 'reg query "HKEY_CURRENT_USER\\Software\\Google\\Chrome\\BLBeacon" /v version'
        result = subprocess.run(command, capture_output=True, text=True, check=True, shell=True)
        # The version is usually the last word in the second line of output
        version_line = [line for line in result.stdout.split('\n') if 'version' in line]
        if version_line:
            return version_line[0].split()[-1].strip()
        return "Unknown (Registry read failed)"
    except Exception:
        return "Unknown (Registry or subprocess error)"

def scrape_bricklink_sales():
    """
    Initializes a Chrome browser (headless), navigates to the item page,
    clicks the 'Price Guide' link (to expand the sales data), extracts the 6-month sales data, and saves it to a CSV.
    """
    # Set up Chrome options for running headless (without a visible GUI)
    options = webdriver.ChromeOptions()
    options.add_argument("--headless") # Run in background
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")

    print("Starting Chrome Driver...")
    
    # --- DRIVER INITIALIZATION ---
    # If the automatic driver management or system PATH setup fails, you MUST provide 
    # the explicit, full path to your downloaded chromedriver.exe here.
    # IMPORTANT: Use a RAW STRING (r"...") to correctly handle Windows backslashes.
    # Example: CHROME_DRIVER_PATH = r"C:\SeleniumDrivers\chromedriver.exe"
    CHROME_DRIVER_PATH = r"C:\Pricing_BL\chromedriver.exe"
                            
    try:
        # Check Chrome version using a system command (best effort)
        print(f"Detected Chrome Browser Version: {get_chrome_version_from_registry()}")
        print("Ensure your ChromeDriver matches this major version.")

        if CHROME_DRIVER_PATH:
            # Option 1: Explicitly set the path using the Service object (Required for Selenium 4.6+)
            service = Service(executable_path=CHROME_DRIVER_PATH)
            driver = webdriver.Chrome(options=options, service=service)
            print(f"Using explicit ChromeDriver path: {CHROME_DRIVER_PATH}")
        else:
            # Option 2: Rely on automatic driver management (Selenium Manager is the default)
            driver = webdriver.Chrome(options=options)
            print("Relying on automatic driver detection (Selenium Manager).")
            # Selenium Manager usually downloads the driver to:
            # C:\Users\<YourUser>\AppData\Local\SeleniumManager\
            # If the script fails here, the driver either wasn't downloaded or is incompatible.

        # ---------------------------
        
        driver.get(ITEM_URL)
        print(f"Navigated to: {ITEM_URL}")

        # Wait for the page to load completely 
        WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.TAG_NAME, "body"))
        )

        # 1. Attempt to click the link to view the Price Guide (which expands the sales history table)
        try:
            print("Looking for the 'View Price Guide' link to expand sales data...")
            
            # Locate the "View Price Guide" link/button
            price_guide_link = WebDriverWait(driver, 15).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, PRICE_GUIDE_LINK_SELECTOR))
            )
            price_guide_link.click()
            print("Successfully clicked to load/expand sales data.")

        except TimeoutException:
            print("Could not find or click the initial price guide link using CSS selector. Check selector.")
            print("The data might be loaded without a click, proceeding to table extraction.")
            # We continue here in case the data is sometimes visible without the explicit click

        # 2. Wait for the 'Past 6 Months Sales' table content to be visible
        print("Waiting for sales data table to load...")
        sales_table = WebDriverWait(driver, 20).until(
            EC.presence_of_element_located((By.XPATH, SALES_TABLE_XPATH))
        )
        print("Sales data table is now visible.")

        # 3. Extract the data
        sales_rows = sales_table.find_elements(By.TAG_NAME, "tr")

        if not sales_rows:
            print("No sales data rows found in the table.")
            return

        # Prepare data for CSV
        extracted_data = []

        # Process the header row (assuming first row)
        header_cells = sales_rows[0].find_elements(By.TAG_NAME, "th")
        header = [cell.text.strip() for cell in header_cells if cell.text.strip()]
        if not header:
             # Fallback header if primary extraction fails
             header = ["Date", "Quantity", "New/Used", "Price/Unit", "Total Price", "Country"]

        extracted_data.append(header)
        print(f"Header found: {header}")

        # Process the remaining rows (data rows)
        for row in sales_rows[1:]:
            try:
                # Find all data cells (td) in the row
                cells = row.find_elements(By.TAG_NAME, "td")
                row_data = [cell.text.strip() for cell in cells]
                
                # Basic validation: ensure the row has a reasonable number of columns
                if len(row_data) > 3:
                    extracted_data.append(row_data)
            except Exception as e:
                # Catch any parsing error for a specific row and continue
                print(f"Error processing row: {e}")
                continue

        # 4. Save to CSV
        if len(extracted_data) > 1: # Check if we have header + at least one data row
            with open(OUTPUT_FILE, 'w', newline='', encoding='utf-8') as f:
                writer = csv.writer(f)
                writer.writerows(extracted_data)

            print(f"\n--- SUCCESS ---")
            print(f"Extracted {len(extracted_data) - 1} sales records for {ITEM_NAME}.")
            print(f"Data saved to {OUTPUT_FILE}")
        else:
            print("Extraction finished, but no sales records were found.")

    except WebDriverException as e:
        print(f"\n--- AN ERROR OCCURRED ---")
        print(f"Error: {e.msg}")
        print("\n--- RESOLUTION CHECKLIST ---")
        print("1. **Browser/Driver Mismatch:** Is your installed Chrome browser version (see output above) compatible with the ChromeDriver being used?")
        print("2. **Explicit Path Required:** If you are getting a long stack trace, uncomment and correctly set `CHROME_DRIVER_PATH` to the exact file path of your compatible `chromedriver.exe` (using the `r''` prefix).")
        print("3. **Automatic Path (Selenium Manager):** If you rely on automatic detection, check if the driver was downloaded to the default location (e.g., `C:\\Users\\<YourUser>\\AppData\\Local\\SeleniumManager\\`).")

    except Exception as e:
        print(f"\n--- AN UNEXPECTED ERROR OCCURRED ---")
        print(f"Error: {e}")

    finally:
        # 5. Clean up: close the browser
        if 'driver' in locals():
            driver.quit()

# Execute the scraper function
if __name__ == "__main__":
    scrape_bricklink_sales()



Starting Chrome Driver...
Detected Chrome Browser Version: 142.0.7444.176
Ensure your ChromeDriver matches this major version.

--- AN ERROR OCCURRED ---
Error: Can not connect to the Service C:\Windows\SeleniumDrivers\chrome-win64\chrome.exe

--- RESOLUTION CHECKLIST ---
1. **Browser/Driver Mismatch:** Is your installed Chrome browser version (see output above) compatible with the ChromeDriver being used?
2. **Explicit Path Required:** If you are getting a long stack trace, uncomment and correctly set `CHROME_DRIVER_PATH` to the exact file path of your compatible `chromedriver.exe` (using the `r''` prefix).
3. **Automatic Path (Selenium Manager):** If you rely on automatic detection, check if the driver was downloaded to the default location (e.g., `C:\Users\<YourUser>\AppData\Local\SeleniumManager\`).


In [4]:
"C:\Windows\SeleniumDrivers\chrome-win64\chrome.exe"

!google-chrome --version

<>:1: SyntaxWarning: invalid escape sequence '\W'
<>:1: SyntaxWarning: invalid escape sequence '\W'
C:\Users\sezar\AppData\Local\Temp\ipykernel_7120\1819534532.py:1: SyntaxWarning: invalid escape sequence '\W'
  "C:\Windows\SeleniumDrivers\chrome-win64\chrome.exe"
'google-chrome' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
import sys
!chromedriver --version



'chromedriver' is not recognized as an internal or external command,
operable program or batch file.


In [7]:
!pip install selenium

You should consider upgrading via the 'c:\users\sezar\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.


In [1]:
!pip install --upgrade ipykernel

  Attempting uninstall: ipykernel
    Found existing installation: ipykernel 6.30.0
    Uninstalling ipykernel-6.30.0:
      Successfully uninstalled ipykernel-6.30.0


ERROR: After October 2020 you may experience errors when installing or updating packages. This is because pip will change the way that it resolves dependency conflicts.

We recommend you use --use-feature=2020-resolver to test your packages with the new resolver before it becomes the default.

notebook 6.4.12 requires ipython-genutils, which is not installed.
notebook 6.4.12 requires Send2Trash>=1.8.0, which is not installed.
nbclassic 0.4.4 requires ipython-genutils, which is not installed.
nbclassic 0.4.4 requires Send2Trash>=1.8.0, which is not installed.
You should consider upgrading via the 'c:\users\sezar\.pyenv\pyenv-win\versions\3.9.0\python.exe -m pip install --upgrade pip' command.
